In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import os
from datetime import datetime
from IPython.display import display, HTML

# [cite: 2026-01-11] 静态快照与代码格式化防御机制
# 确保在闭市期间秒开，在交易时间自动更新
SNAPSHOT_FILE = "market_snapshot.csv"

def get_optimized_data_v61():
    now = datetime.now()
    # 闭市判断逻辑 (15:30 以后至次日 09:15 之前)
    is_market_closed = now.hour >= 16 or now.hour < 9 or (now.hour == 15 and now.minute > 30)
    
    if is_market_closed and os.path.exists(SNAPSHOT_FILE):
        return pd.read_csv(SNAPSHOT_FILE, dtype={'代码': str}) 
    else:
        try:
            df = ak.stock_zh_a_spot_em()
            if df is not None:
                # [cite: 2026-01-11] 格式化防御：补齐 6 位代码的前导零
                df['代码'] = df['代码'].astype(str).str.zfill(6)
                df.to_csv(SNAPSHOT_FILE, index=False, encoding='utf_8_sig')
                return df
        except Exception as e:
            print(f"📡 实时抓取提示: {e}，尝试调用本地旧快照...")
            return pd.read_csv(SNAPSHOT_FILE, dtype={'代码': str}) if os.path.exists(SNAPSHOT_FILE) else None

def run_alpha_v61_terminal():
    df_raw = get_optimized_data_v61()
    if df_raw is None: 
        print("❌ 未能获取数据，请检查网络或快照文件。")
        return

    # 1. 自动寻找列名映射 (处理不同接口版本差异)
    def get_c(cands): return next((c for c in cands if c in df_raw.columns), None)
    c_p, c_c, c_o, c_h, c_l = get_c(['涨跌幅','涨幅']), get_c(['最新价','收盘']), get_c(['开盘','今开']), get_c(['最高']), get_c(['最低'])
    c_n, c_id = get_c(['名称']), get_c(['代码'])

    # 2. 数据清洗 [cite: 2026-01-11]
    df = df_raw.copy()
    for col in [c_p, c_c, c_o, c_h, c_l]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 3. Alpha001 核心逻辑计算
    # 公式：-1 * rank(rank(close - open) / rank(high - low)) 简化实证版
    df['rank_val'] = df[c_p].argsort().argsort() / len(df)
    df['sup'] = df.apply(lambda r: (min(r[c_o], r[c_c]) - r[c_l]) / (r[c_h] - r[c_l]) if (r[c_h]-r[c_l]) > 0.001 else 0, axis=1)
    
    # 筛选 Top 50
    final_list = df[df['sup'] >= 0.60].sort_values('rank_val', ascending=False).head(50)

    # 4. 自动备份今日成果 [cite: 2026-01-11]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    final_list.to_csv(f"Alpha001_Result_{timestamp}.csv", index=False, encoding='utf_8_sig')

    # 5. UI 视觉渲染 (Bloomberg Dark 风格)
    C_BG, C_LABEL, C_GOLD, C_ENERGY = "#0d1117", "#c9d1d9", "#d4b77d", "#f86b6b"
    today_str = datetime.now().strftime("%Y年1月13日")
    
    # 渲染大标题：像素级对齐，词组微气口
    display(HTML(f"""
        <div style="max-width:650px; padding:8px 0 8px 28px; margin-bottom:10px; border-left:3px solid {C_GOLD};">
            <h1 style="color:{C_GOLD}; font-family:sans-serif; margin:0; font-size:25px; font-weight:bold;">
                全市场<span style="margin:0 0.4em;">Alpha001</span>实证<span style="margin:0 0.4em;">50</span>
                <span style="color:{C_LABEL}; font-size:17px; font-weight:normal; margin-left:6px;">({today_str})</span>
            </h1>
        </div>
    """))

    for i, (idx, row) in enumerate(final_list.iterrows(), 1):
        s_n, s_id, s_p, s_c = row[c_n], str(row[c_id]).zfill(6), row[c_p], row[c_c]
        
        display(HTML(f"""
        <div style="background:{C_BG}; color:#ffffff; padding:18px 28px; border-radius:4px; border:1px solid #30363d; margin-bottom:4px; font-family:sans-serif; max-width:650px;">
            <div style="display:flex; align-items:center;">
                <div style="flex:1; display:flex; align-items:baseline;">
                    <span style="color:{C_GOLD}; font-size:24px; font-family:monospace; font-weight:bold; width:36px;">{i:02d}</span> 
                    <span style="color:{C_GOLD}; font-size:24px; font-weight:bold; margin-left:1.5px;">{s_n}</span>
                    <span style="margin-left:10px; font-size:18px; color:{C_LABEL}; opacity:0.8;">({s_id})</span>
                </div>
                <div style="width:1px; height:24px; background:rgba(255,255,255,0.25); margin:0 25px;"></div>
                <div style="flex:1; font-size:24px; font-weight:bold; color:{C_ENERGY}; display:flex; align-items:baseline;">
                    <span style="min-width:90px;">¥{s_c:.2f}</span>
                    <span style="font-family:monospace; margin-left:6px;">{s_p:+.2f}%</span>
                </div>
            </div>
            <div style="height:1px; background:#21262d; margin:12px 0;"></div>
            <div style="display:flex; align-items:stretch;">
                <div style="flex:1;">
                    <div style="color:{C_LABEL}; font-size:12px; font-weight:600; margin-bottom:6px;">全市场 Alpha001 Rank</div>
                    <div style="font-size:24px; font-weight:bold; color:{C_GOLD}; font-family:monospace; line-height:1;">{row['rank_val']:.6f}</div>
                </div>
                <div style="width:1px; background:rgba(255,255,255,0.25); margin:0 25px;"></div>
                <div style="flex:1;">
                    <div style="color:{C_LABEL}; font-size:12px; font-weight:600; margin-bottom:6px;">托盘力度 / 下影线占比</div>
                    <div style="font-size:28px; font-weight:bold; color:{C_ENERGY}; line-height:1; margin-bottom:10px;">
                        {row['sup']*100:.1f}<span style="font-size:16px; margin-left:1px;">%</span>
                    </div>
                    <div style="background:{C_LABEL}33; height:4px; border-radius:2px; width:100%; overflow:hidden;">
                        <div style="background:{C_ENERGY}; width:{row['sup']*100}%; height:100%;"></div>
                    </div>
                </div>
            </div>
        </div>
        """))

if __name__ == "__main__":
    run_alpha_v61_terminal()